# SQL 1: basics

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-01-basics.ipynb)

SQL (Structured Query Language) is how you ask a database questions. You describe the rows you want, and the
engine works out how to find them. This notebook covers the everyday building blocks: choosing columns,
filtering rows, sorting, summarising, grouping, combining tables, conditional values and missing values.

The examples run in Spark SQL, the SQL dialect of Apache Spark. Most of what you learn here works unchanged in
PostgreSQL, MySQL, BigQuery, Snowflake and Redshift. The notebook ends with practice questions and their answers.

## Setup

The first cell installs PySpark when it is missing, which takes about a minute in Colab. The second starts a Spark session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## The example data

Two small tables, `customers` and `orders`, are created below as temporary views. A view is a name for a table
that lives only for this session, so you can query it like any table.

`customers` has one row per customer. `orders` has one row per order, and `customer_id` links each order to its
customer. Customers 7 and 8 have never ordered, which the later examples rely on.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

The `orders` table has 14 rows: 11 `completed`, 2 `cancelled` and 1 `pending`. Every `amount` is a different
number, so a sort by amount never has ties. The cell below checks those facts, so later answers can rely on them.

In [ ]:
assert spark.sql("SELECT COUNT(*) FROM customers").collect() == [(8,)]
assert spark.sql("SELECT COUNT(*) FROM orders").collect() == [(14,)]
assert spark.sql(
    "SELECT status, COUNT(*) FROM orders GROUP BY status ORDER BY status"
).collect() == [("cancelled", 2), ("completed", 11), ("pending", 1)]
assert spark.sql("SELECT COUNT(DISTINCT amount) FROM orders").collect() == [(14,)]

## 1. Tables, rows and columns

A **table** is a set of rows that share the same columns. Each **row** describes one thing, here one customer or
one order. Each **column** holds one attribute of that thing, such as `name` or `amount`. The list of column
names and their types is the table's **schema**.

What one row represents is called the table's **grain**. The grain of `customers` is one customer; the grain of
`orders` is one order. Knowing the grain tells you what a count of rows means.

The simplest query reads every row and every column:

```sql
SELECT * FROM customers;
```

`SELECT` names the columns you want, `FROM` names the table, and `*` means "all columns". The rest of this
notebook replaces the `*` with what you actually need.

SQL is **declarative**: you state what result you want, not the steps to compute it. The engine chooses how to
read the data. The result of every query is itself a table, so it can be read by another query.

### Where you'll use it

- **Every warehouse table:** a fact or dimension table in Snowflake, BigQuery or Redshift is exactly this: rows with named, typed columns.
- **Raw landing tables:** the first thing a pipeline does with a file is treat it as a table of rows and columns.
- **Checking the grain:** before you join or count, confirm what one row of each table means.

In [ ]:
result = spark.sql("SELECT * FROM customers")
result.show()

assert result.count() == 8
assert result.columns == ["customer_id", "name", "region", "signup_date"]

## 2. Choosing columns

List the columns you want, separated by commas. Give a column a shorter or clearer name with `AS`; that name is
an **alias**. A column can also be an expression, such as `amount * 2`.

```sql
SELECT column1, column2 AS alias FROM table;
SELECT DISTINCT column FROM table;
```

### Where you'll use it

- **Narrow reads:** selecting only the needed columns is how columnar engines avoid reading the rest of the data, which cuts cost on pay-per-scan warehouses.
- **Dashboard extracts:** a report query returns the four columns the chart needs, not `SELECT *`.

In [ ]:
spark.sql("""
SELECT name, region
FROM customers
""").show()

aliased = spark.sql("""
SELECT name AS customer_name, customer_id * 100 AS scaled_id
FROM customers
""")
aliased.show(3)
assert aliased.columns == ["customer_name", "scaled_id"]
assert tuple(aliased.orderBy("scaled_id").first()) == ("Acme Corp", 100)

### DISTINCT works on whole rows

`DISTINCT` removes duplicate rows from the result. Two rows are duplicates only when **every** selected column is
equal. With one column, `region`, the eight customers collapse to four regions. With two columns,
`customer_id, status`, a row is kept for each different pair:

| customer_id | statuses in `orders` | distinct pairs |
| --- | --- | --- |
| 1 | completed x3 | 1 |
| 2 | completed x2, cancelled | 2 |
| 3 | completed x2 | 1 |
| 4 | pending, completed | 2 |
| 5 | completed x2 | 1 |
| 6 | completed, cancelled | 2 |

That gives 9 distinct pairs from 14 orders.

In [ ]:
q = "SELECT DISTINCT region FROM customers ORDER BY region"
spark.sql(q).show()
assert spark.sql(q).collect() == [("East",), ("North",), ("South",), ("West",)]

pairs = spark.sql("SELECT DISTINCT customer_id, status FROM orders")
assert pairs.count() == 9

## 3. Filtering rows with WHERE

`WHERE` keeps only the rows whose condition is true. The condition is called a **predicate**: an expression that
is true, false or unknown for each row. Only rows where it is true pass. `WHERE` runs before grouping and before
`SELECT`, so it decides which rows the rest of the query sees.

| Operator | Meaning |
| --- | --- |
| `=`, `<>` | equal, not equal |
| `<`, `<=`, `>`, `>=` | comparisons |
| `AND`, `OR`, `NOT` | combine conditions |
| `IN (...)` | equal to any value in the list |
| `BETWEEN a AND b` | from a to b, both ends included |
| `LIKE` | text pattern: `%` is any text, `_` is one character |

`LIKE` is case-sensitive in Spark: `'Acme' LIKE 'acme%'` is false.

### Where you'll use it

- **Date windows:** `WHERE order_date >= ...` restricts a daily load or a month-end report.
- **Data-quality rules:** finding rows that break a rule, such as `WHERE amount < 0`.

Each query below states its expected rows in the `assert`. Work them out from the `orders` table first.

- Completed and at least 300: orders 4 (300), 6 (400), 10 (320), 11 (500) and 12 (450).
- `BETWEEN 150 AND 300`: orders 2, 3, 4, 5, 8, 9 and 13. Order 3 (150) and order 4 (300) are kept, which shows
  both ends are included.

In [ ]:
q = """
SELECT order_id, amount
FROM orders
WHERE status = 'completed' AND amount >= 300
ORDER BY order_id
"""
spark.sql(q).show()
assert [row.order_id for row in spark.sql(q).collect()] == [4, 6, 10, 11, 12]

q = """
SELECT name, region
FROM customers
WHERE region IN ('North', 'West')
ORDER BY name
"""
spark.sql(q).show()
assert [row.name for row in spark.sql(q).collect()] == ["Acme Corp", "Beta LLC", "Echo Ltd", "Foxtrot GmbH"]

q = """
SELECT order_id, amount
FROM orders
WHERE amount BETWEEN 150 AND 300
ORDER BY order_id
"""
spark.sql(q).show()
assert [row.order_id for row in spark.sql(q).collect()] == [2, 3, 4, 5, 8, 9, 13]

q = """
SELECT name
FROM customers
WHERE name LIKE '%Corp' OR name LIKE 'Echo%'
ORDER BY name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Acme Corp",), ("Echo Ltd",)]

### AND binds tighter than OR

`AND` is evaluated before `OR`, the way multiplication is evaluated before addition. So

```sql
status = 'cancelled' OR status = 'pending' AND amount > 300
```

means `cancelled OR (pending AND amount > 300)`. Both cancelled orders (5 and 14) pass. The only pending order,
9, has amount 220, so it fails. Adding parentheses changes the meaning:
`(cancelled OR pending) AND amount > 300` keeps only order 14 (325). Use parentheses whenever you mix the two.

In [ ]:
without_parentheses = spark.sql("""
SELECT order_id FROM orders
WHERE status = 'cancelled' OR status = 'pending' AND amount > 300
ORDER BY order_id
""").collect()

with_parentheses = spark.sql("""
SELECT order_id FROM orders
WHERE (status = 'cancelled' OR status = 'pending') AND amount > 300
ORDER BY order_id
""").collect()

print(without_parentheses, with_parentheses)
assert without_parentheses == [(5,), (14,)]
assert with_parentheses == [(14,)]

## 4. Sorting and limiting

`ORDER BY` sorts the result. `ASC` is ascending (the default) and `DESC` is descending. You can sort by several
columns; later columns break ties in earlier ones. `LIMIT n` returns only the first `n` rows, which is useful
together with a sort.

```sql
SELECT ... FROM table ORDER BY column1 DESC, column2 ASC LIMIT 10;
```

A table has no built-in row order. Without `ORDER BY`, the engine may return rows in any order, and the order
can change between runs. `LIMIT` without `ORDER BY` therefore returns an arbitrary set of rows.

### Where you'll use it

- **Top-N reports:** the ten best-selling products for a dashboard.
- **Spot checks:** `ORDER BY ... LIMIT 20` to look at the newest rows after a load.

In [ ]:
q = """
SELECT order_id, amount
FROM orders
ORDER BY amount DESC
LIMIT 3
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(11, 500), (12, 450), (6, 400)]

### Ties need a tie-breaker

Rows with equal sort values are called **peers**. Their order among themselves is not defined. Customer 1 has
three orders, so `ORDER BY customer_id` alone may list orders 1, 2 and 3 in any order. Adding a unique column,
such as `order_id`, as the last sort key makes the order fully determined.

In [ ]:
q = """
SELECT customer_id, order_id
FROM orders
ORDER BY customer_id, order_id
LIMIT 4
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 1), (1, 2), (1, 3), (2, 4)]

### Where NULLs sort

In Spark, `NULL` values sort first in ascending order and last in descending order. Add `NULLS LAST` or
`NULLS FIRST` to choose explicitly. Engines differ here: PostgreSQL and Oracle treat `NULL` as larger than any
value, so they put it last in ascending order. Write the clause when the position matters.

In [ ]:
scores = "SELECT * FROM VALUES (1, 10), (2, NULL), (3, 30) AS scores(id, score)"

ascending = spark.sql(f"SELECT score FROM ({scores}) ORDER BY score").collect()
nulls_last = spark.sql(f"SELECT score FROM ({scores}) ORDER BY score NULLS LAST").collect()
descending = spark.sql(f"SELECT score FROM ({scores}) ORDER BY score DESC").collect()

print(ascending, nulls_last, descending)
assert ascending == [(None,), (10,), (30,)]
assert nulls_last == [(10,), (30,), (None,)]
assert descending == [(30,), (10,), (None,)]

## 5. Summarising with aggregate functions

An **aggregate function** turns many rows into one value.

| Function | Returns |
| --- | --- |
| `COUNT(*)` | number of rows |
| `COUNT(column)` | number of rows where `column` is not `NULL` |
| `SUM(column)` | total |
| `AVG(column)` | average |
| `MIN(column)`, `MAX(column)` | smallest and largest value |

Without a `GROUP BY` (next section), the whole table is one group and the query returns one row.

### Where you'll use it

- **KPIs:** total revenue, distinct active users and average order value are aggregates over a fact table.
- **Reconciliation:** comparing `COUNT(*)` and `SUM(amount)` between a source and the warehouse after a load.

The 11 completed orders add up to 2,995. Their average is 2995 / 11 = 272.2727..., which rounds to 272.27. The
smallest is order 1 (100) and the largest is order 11 (500). `AVG` returns a `DOUBLE`, so the check compares
floats.

In [ ]:
q = """
SELECT
  COUNT(*)                  AS orders,
  SUM(amount)               AS total_amount,
  ROUND(AVG(amount), 2)     AS average_amount,
  MIN(amount)               AS smallest,
  MAX(amount)               AS largest
FROM orders
WHERE status = 'completed'
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(11, 2995, 272.27, 100, 500)]

### Aggregates skip NULLs

`SUM`, `AVG`, `MIN`, `MAX` and `COUNT(column)` ignore `NULL` values. Only `COUNT(*)` counts every row. Take three
scores: 10, `NULL` and 30.

| Expression | Works on | Result |
| --- | --- | --- |
| `COUNT(*)` | 3 rows | 3 |
| `COUNT(score)` | 10, 30 | 2 |
| `SUM(score)` | 10, 30 | 40 |
| `AVG(score)` | 10, 30 | 20.0, not 40 / 3 |

If no rows reach the aggregate at all, `COUNT` returns 0 but `SUM`, `AVG`, `MIN` and `MAX` return `NULL`.

In [ ]:
q = """
SELECT COUNT(*) AS all_rows, COUNT(score) AS scored, SUM(score) AS total, AVG(score) AS average
FROM VALUES (1, 10), (2, NULL), (3, 30) AS scores(id, score)
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(3, 2, 40, 20.0)]

q = "SELECT COUNT(*) AS orders, SUM(amount) AS total FROM orders WHERE status = 'refunded'"
spark.sql(q).show()
assert spark.sql(q).collect() == [(0, None)]

## 6. Grouping with GROUP BY

`GROUP BY` splits the rows into groups and computes the aggregates once per group. Rows with equal values in the
`GROUP BY` columns go into the same group. The result has one row per group, so the `GROUP BY` columns become
the grain of the result.

```sql
SELECT column, COUNT(*) FROM table GROUP BY column;
```

Grouping `orders` by `status` forms three groups:

| status | orders in the group | COUNT(*) | SUM(amount) |
| --- | --- | --- | --- |
| cancelled | 5, 14 | 2 | 250 + 325 = 575 |
| completed | 1, 2, 3, 4, 6, 7, 8, 10, 11, 12, 13 | 11 | 2,995 |
| pending | 9 | 1 | 220 |

### Where you'll use it

- **Every metric by dimension:** revenue by region, by month or by product category.
- **Deduplicating by key:** `GROUP BY order_id` to find keys that appear more than once.

In [ ]:
q = """
SELECT status, COUNT(*) AS orders, SUM(amount) AS total_amount
FROM orders
GROUP BY status
ORDER BY status
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("cancelled", 2, 575), ("completed", 11, 2995), ("pending", 1, 220)]

Grouping by two columns makes one group per **combination** of values. These are the nine pairs counted in the
`DISTINCT` example of section 2, now with a count for each.

In [ ]:
q = """
SELECT customer_id, status, COUNT(*) AS orders
FROM orders
GROUP BY customer_id, status
ORDER BY customer_id, status
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (1, "completed", 3),
    (2, "cancelled", 1), (2, "completed", 2),
    (3, "completed", 2),
    (4, "completed", 1), (4, "pending", 1),
    (5, "completed", 2),
    (6, "cancelled", 1), (6, "completed", 1),
]

### Every selected column must be grouped or aggregated

Each output row stands for a whole group. A column that is neither in `GROUP BY` nor inside an aggregate has
many values in the group, and the engine cannot pick one. Selecting `order_id` next to `GROUP BY status` is
therefore rejected when the query is analysed, before any data is read.

In [ ]:
from pyspark.errors import AnalysisException

rejected = False
try:
    spark.sql("SELECT status, order_id FROM orders GROUP BY status")
except AnalysisException as error:
    rejected = True
    print(str(error)[:90])
assert rejected

## 7. WHERE vs HAVING

`WHERE` and `HAVING` both filter, but they run at different stages. `WHERE` filters **rows** before anything is
grouped. `HAVING` filters **groups** after the aggregates are computed.

| Question | Keyword |
| --- | --- |
| Is this one order completed? | `WHERE status = 'completed'` |
| Does this customer's total exceed 500? | `HAVING SUM(amount) > 500` |
| Does this customer have more than one completed order? | `WHERE` for the status, `HAVING COUNT(*) > 1` |

A practical test: if the condition is about a single order, it goes in `WHERE`. If it is about a customer's total
or count, it goes in `HAVING`.

### Where you'll use it

- **Filtering on a metric:** "customers who spent more than 1,000" needs `HAVING SUM(amount) > 1000`.
- **Finding duplicate keys:** `GROUP BY order_id HAVING COUNT(*) > 1` lists keys loaded twice.

Completed totals per customer are 1: 450, 2: 700, 3: 300, 4: 320, 5: 950 and 6: 275. Only customers 2 and 5
are above 500.

In [ ]:
q = """
SELECT customer_id, SUM(amount) AS completed_total
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
HAVING SUM(amount) > 500
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, 700), (5, 950)]

The `status` condition is about each order, so it stays in `WHERE`. The threshold is about each customer's total, so
it goes in `HAVING`.

Filtering rows first also changes the answer, because rows removed by `WHERE` never reach the aggregates. Drop
the `WHERE` and the totals include cancelled and pending orders: customer 4 becomes 220 + 320 = 540 and
customer 6 becomes 275 + 325 = 600, so both now pass the threshold.

In [ ]:
q = """
SELECT customer_id, SUM(amount) AS any_status_total
FROM orders
GROUP BY customer_id
HAVING SUM(amount) > 500
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, 950), (4, 540), (5, 950), (6, 600)]

**Common mistakes.** `WHERE SUM(amount) > 500` is rejected, because the sum does not exist yet at the `WHERE` stage.
`HAVING status = 'completed'` fails, because `status` is neither grouped nor aggregated. Move each condition to the
stage it belongs to.

In [ ]:
rejected = False
try:
    spark.sql("SELECT customer_id FROM orders WHERE SUM(amount) > 500 GROUP BY customer_id")
except AnalysisException as error:
    rejected = True
    print(str(error)[:90])
assert rejected

## 8. How a query runs

You write a query in one order, but the engine evaluates its clauses in another:

1. `FROM` and `JOIN`: choose the tables and combine them.
2. `WHERE`: keep the rows that match.
3. `GROUP BY`: form the groups.
4. `HAVING`: keep the groups that match.
5. `SELECT`: compute the output columns, then `DISTINCT` removes duplicates.
6. `ORDER BY`: sort the result.
7. `LIMIT`: keep the first rows.

This is the **logical** order: it defines what the answer is. The optimizer may do the physical work in a
different order, for example by reading only the rows a filter needs, but the result is always the same as if
the steps ran in this order.

### Where you'll use it

- **Performance tuning:** reading the plan (`EXPLAIN`) shows whether a filter runs early and whether a join scans a full table.
- **Debugging errors:** "column not found" for an alias usually means the alias was used in a clause that runs before `SELECT`.

The order explains two rules from earlier sections. An alias defined in `SELECT` cannot be used in `WHERE`,
because `WHERE` runs first. An alias can be used in `ORDER BY`, because `ORDER BY` runs after `SELECT`. For the
same reason an aggregate cannot appear in `WHERE`.

In [ ]:
rejected = False
try:
    spark.sql("SELECT order_id, amount * 2 AS doubled FROM orders WHERE doubled > 800")
except AnalysisException as error:
    rejected = True
    print(str(error)[:90])
assert rejected

q = "SELECT order_id, amount * 2 AS doubled FROM orders ORDER BY doubled DESC LIMIT 2"
spark.sql(q).show()
assert spark.sql(q).collect() == [(11, 1000), (12, 900)]

## 9. Combining tables with JOIN

A **join** matches rows from two tables by a **key**. Here `orders.customer_id` points to `customers.customer_id`,
and the `ON` clause says which rows match. This section is a short preview so the practice questions can use joins;
[SQL 2: joins](sql-02-joins.ipynb) covers them in full.

- `INNER JOIN` (or just `JOIN`) keeps only the pairs of rows that match.
- `LEFT JOIN` keeps every row of the left table, with `NULL` in the right-hand columns when there is no match.

```sql
SELECT ... FROM left_table l JOIN right_table r ON l.key = r.key;
```

### Where you'll use it

- **Enriching facts:** attaching customer names and product categories to an orders table.
- **Matching sources:** joining a bank feed to the ledger on transaction ID to find unmatched payments.

### Row counts tell you what a join did

The inner join returns 14 rows, one per order: the result has the grain of `orders`. The left join from `customers`
adds one row each for customers 7 and 8, who have no orders: 16 rows.

In [ ]:
inner = spark.sql("""
SELECT o.order_id, c.name, o.amount
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
ORDER BY o.order_id
""")
inner.show(4)
assert inner.count() == 14
assert inner.filter("name = 'Acme Corp'").count() == 3

left = spark.sql("""
SELECT c.name, o.order_id
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
""")
assert left.count() == 16

### Finding rows with no match

Left join and keep the rows where the right-hand key (`order_id`) is `NULL`: that can only mean "no match".

In [ ]:
q = """
SELECT c.name, o.order_id
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL
ORDER BY c.name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Gamma SA", None), ("Hotel BV", None)]

## 10. Conditional values with CASE

`CASE` is SQL's if/else: it returns the value of the first true branch, or `NULL` when nothing matches and there is
no `ELSE`. This is a short preview; [SQL 4: CASE expressions](sql-04-case-expressions.ipynb) covers branch order,
the simple form and the common mistakes.

```sql
CASE WHEN condition THEN value WHEN condition2 THEN value2 ELSE other END
```

### Where you'll use it

- **Labels for dashboards:** mapping status codes to readable names or buckets such as "small", "medium", "large".
- **Pivots:** turning rows into columns with `SUM(CASE WHEN ...)`.

In [ ]:
q = """
SELECT order_id, amount,
       CASE WHEN amount >= 300 THEN 'big'
            ELSE 'small'
       END AS size
FROM orders
ORDER BY order_id
LIMIT 5
"""
spark.sql(q).show()
assert [row.size for row in spark.sql(q).collect()] == ["small", "small", "small", "big", "small"]

### CASE inside an aggregate

`SUM(CASE WHEN status = 'completed' THEN amount ELSE 0 END)` adds up only the rows you choose, so one pass over the
table gives each customer's completed and cancelled totals as two columns. This is **conditional aggregation**.

In [ ]:
q = """
SELECT customer_id,
       SUM(CASE WHEN status = 'completed' THEN amount ELSE 0 END) AS completed_total,
       SUM(CASE WHEN status = 'cancelled' THEN amount ELSE 0 END) AS cancelled_total
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (1, 450, 0), (2, 700, 250), (3, 300, 0), (4, 320, 0), (5, 950, 0), (6, 275, 325),
]

## 11. Missing values with NULL

`NULL` means the value is unknown. It is not zero and it is not an empty string. Test for missing values with
`IS NULL` and `IS NOT NULL`, and supply a default with `COALESCE(a, b)`, which returns the first value that is not
`NULL`. This is a short preview; [SQL 3: NULL handling](sql-03-null-handling.ipynb) covers NULLs in filters, joins,
aggregates and sorting, NULL-safe comparison and the `NOT IN` trap.

### Where you'll use it

- **Optional fields:** a shipping date that is null until the order ships.
- **Late data:** a join that returns NULL on the right side tells you a record has not arrived yet.

In [ ]:
q = """
WITH scores AS (
  SELECT * FROM VALUES (1, 10), (2, NULL), (3, 30) AS scores(id, score)
)
SELECT id,
       score,
       score IS NULL            AS is_missing,
       COALESCE(score, 0)       AS score_or_zero
FROM scores
ORDER BY id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 10, False, 10), (2, None, True, 0), (3, 30, False, 30)]

### Comparisons with NULL are unknown

Any comparison with `NULL` is unknown, not true or false, and arithmetic with `NULL` is `NULL`. `WHERE` keeps only
rows where the condition is **true**, so `WHERE x = NULL` matches nothing and `WHERE score <> 10` drops the row whose
score is `NULL`. [SQL 3](sql-03-null-handling.ipynb) works through this three-valued logic.

In [ ]:
assert spark.sql("SELECT NULL = NULL, NULL + 1, NULL AND false, NULL OR true").collect() == [
    (None, None, False, True)
]

q = """
SELECT id FROM VALUES (1, 10), (2, NULL), (3, 30) AS scores(id, score)
WHERE score <> 10
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(3,)]

## Interview notes

- **COUNT(*) vs COUNT(column).** `COUNT(*)` counts rows; `COUNT(column)` counts the non-NULL values in that column.
  The two give different answers whenever the column has NULLs.
- **Empty input.** Over zero rows, `COUNT` returns 0 and `SUM`, `AVG`, `MIN`, `MAX` return `NULL`.
- **WHERE vs HAVING.** Put a row condition in `WHERE`, and a condition on an aggregate in `HAVING`. Interviewers
  often ask which runs first: `WHERE`.
- **Logical order.** `FROM`, `WHERE`, `GROUP BY`, `HAVING`, `SELECT`, `ORDER BY`, `LIMIT`. It explains why a
  `SELECT` alias works in `ORDER BY` but not in `WHERE`.
- **NULL in comparisons.** `x = NULL` is never true; use `IS NULL`.
  [SQL 3: NULL handling](sql-03-null-handling.ipynb) covers NULL-safe `<=>` and `NOT IN` with NULLs.
- **Deterministic order.** Without `ORDER BY` the row order is undefined; add a unique column to break ties.
- **Joins.** An inner join keeps matches; a left join keeps every left row. [SQL 2: joins](sql-02-joins.ipynb) covers the rest.

## Practice questions

Try each question before you open the answers. Each one uses only what this notebook covers.

1. List the names of the customers in the North region, sorted by name.
2. How many orders did each customer place? Show `customer_id` and the count, sorted by `customer_id`.
3. What is the total amount of all completed orders?
4. Which customers have more than one completed order? Show `customer_id` and the number of completed orders.
5. For each status, show the number of orders and the average amount, rounded to 2 decimals.
6. Show every order with the customer's name. Return `order_id`, `name` and `amount`, sorted by `order_id`.
7. Which customers have never placed an order? Show their names, sorted by name.
8. Label each order `big` if its amount is 300 or more and `small` otherwise. How many orders have each label?
9. What is the largest completed order? Show its `order_id` and `amount`.
10. Which customer has the highest total completed amount? Show `customer_id` and the total.
11. Show every customer's name and number of orders, including customers with none (they should show 0).
    Sort by `customer_id`.
12. For each status, count the orders of 300 or more and the orders below 300, as two columns in one row per
    status.

## Answers

Each answer shows the approach, the query, its result, and an `assert` that checks the result.

### Question 1

Filter rows with `WHERE`, then sort with `ORDER BY`.

In [ ]:
q = """
SELECT name FROM customers WHERE region = 'North' ORDER BY name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Acme Corp",), ("Beta LLC",)]

### Question 2

Group `orders` by customer and count the rows in each group.

In [ ]:
q = """
SELECT customer_id, COUNT(*) AS orders
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 3), (2, 3), (3, 2), (4, 2), (5, 2), (6, 2)]

Customers 7 and 8 do not appear: a `GROUP BY` on `orders` only sees customers who have orders. Questions 7 and 11
handle them.

### Question 3

Filter to completed orders, then aggregate the whole result into one row.

In [ ]:
q = """
SELECT SUM(amount) AS completed_total FROM orders WHERE status = 'completed'
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2995,)]

### Question 4

The status is a row condition (`WHERE`); "more than one" is a group condition (`HAVING`).

In [ ]:
q = """
SELECT customer_id, COUNT(*) AS completed_orders
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 3), (2, 2), (3, 2), (5, 2)]

### Question 5

Group by status. `AVG` returns a double, and `ROUND(..., 2)` keeps it a double: cancelled is (250 + 325) / 2 =
287.5, completed is 2995 / 11 = 272.27, pending is 220.0.

In [ ]:
q = """
SELECT status, COUNT(*) AS orders, ROUND(AVG(amount), 2) AS average_amount
FROM orders
GROUP BY status
ORDER BY status
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("cancelled", 2, 287.5), ("completed", 11, 272.27), ("pending", 1, 220.0)]

### Question 6

An inner join on `customer_id`. Every order has a customer, so the result has one row per order: 14 rows.

In [ ]:
q = """
SELECT o.order_id, c.name, o.amount
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
ORDER BY o.order_id
"""
result = spark.sql(q)
result.show(5)
rows = result.collect()
assert len(rows) == 14
assert rows[0] == (1, "Acme Corp", 100) and rows[-1] == (14, "Foxtrot GmbH", 325)

### Question 7

Left join from `customers` and keep the rows where the order key is `NULL`, meaning no match.

In [ ]:
q = """
SELECT c.name
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL
ORDER BY c.name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Gamma SA",), ("Hotel BV",)]

### Question 8

Compute the label with `CASE` and group by the same expression. Orders of 300 or more are 4, 6, 10, 11, 12 and
14: six big, eight small.

In [ ]:
q = """
SELECT CASE WHEN amount >= 300 THEN 'big' ELSE 'small' END AS size, COUNT(*) AS orders
FROM orders
GROUP BY CASE WHEN amount >= 300 THEN 'big' ELSE 'small' END
ORDER BY size
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("big", 6), ("small", 8)]

### Question 9

Sort completed orders by amount, largest first, and keep one row. Amounts are unique, so there is no tie.

In [ ]:
q = """
SELECT order_id, amount FROM orders WHERE status = 'completed' ORDER BY amount DESC LIMIT 1
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(11, 500)]

### Question 10

Total per customer, sort by the total descending, keep one row. Customer 5 has 500 + 450 = 950; the next is
customer 2 with 700, so there is no tie.

In [ ]:
q = """
SELECT customer_id, SUM(amount) AS completed_total
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
ORDER BY completed_total DESC
LIMIT 1
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(5, 950)]

### Question 11

Left join from `customers` so every customer is kept, then count `o.order_id`, not `*`. For Gamma SA the joined
row has `order_id = NULL`: `COUNT(*)` would count that row and return 1, while `COUNT(o.order_id)` skips the
`NULL` and returns 0.

In [ ]:
q = """
SELECT c.name, COUNT(o.order_id) AS orders
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    ("Acme Corp", 3), ("Beta LLC", 3), ("Cyan Inc", 2), ("Delta Co", 2),
    ("Echo Ltd", 2), ("Foxtrot GmbH", 2), ("Gamma SA", 0), ("Hotel BV", 0),
]

### Question 12

Conditional aggregation: one `SUM(CASE ...)` per column, each adding 1 for the rows it should count. Cancelled
has 325 (big) and 250 (small); completed has 300, 400, 320, 500, 450 (big) and six small; pending has 220.

In [ ]:
q = """
SELECT status,
       SUM(CASE WHEN amount >= 300 THEN 1 ELSE 0 END) AS big_orders,
       SUM(CASE WHEN amount <  300 THEN 1 ELSE 0 END) AS small_orders
FROM orders
GROUP BY status
ORDER BY status
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("cancelled", 1, 1), ("completed", 5, 6), ("pending", 0, 1)]